In [1]:
import os
os.environ["MKL_NUM_THREADS"] = "4" 
os.environ["NUMEXPR_NUM_THREADS"] = "4"  
os.environ["OMP_NUM_THREADS"] = "4" 


import numpy as np
import pandas as pd
import yaml
import pickle

from models.sample import *
from utils.tree_logging import *

import argparse
parser = argparse.ArgumentParser()
parser.add_argument('--y_dist', type = str)
# dataset
parser.add_argument('-d', '--dataset_name', type = str, default = 'abalone')
parser.add_argument('-f', '--fold', type = int, default = 0)
parser.add_argument('--log_dir', type = str, default = './parameters')
# model arch
parser.add_argument('--max_depth', type = int, default = 3)
parser.add_argument('-K', '--T_max', type = int, default = 200)
# update
parser.add_argument('--num_samples', type = int, default = 2000)
parser.add_argument('--step_size', type = float, default = 0.01, help = 'height update step size')
parser.add_argument('--leapfrog_L', type = int, default = 1, help = 'height update leapfrog L')
parser.add_argument('--bg_step_size', type = float, default = 0.01, help = 'bg update leapfrog')
parser.add_argument('--M', type = float, default = 1.)
# hyperparameter
parser.add_argument('--gamma_shape', type = float, default = 2.0, help = 'gamma prior shape parameter')
parser.add_argument('--gamma_scale', type = float, default = 0.01, help = 'gamma prior scale parameter')        # 작을 수록 indicator
parser.add_argument('--c_star', type = float, default = 0.001)
# reg
parser.add_argument('--var_height', type = float, default = 0.1)
parser.add_argument('--nu', type = float, default = 10.)
# ber
parser.add_argument('--const_var', type = float, default = 0.1)
parser.add_argument('--const_step_size', type = float, default = 0.01)

parser.add_argument('--seed', type = int, default = 42)
args = parser.parse_args([])

np.random.seed(args.seed)

ALL_FOLD = False


/usr/local/miniconda3/envs/nine/lib/python3.9/site-packages/pandas/core/computation/expressions.py:21: UserWarning: Pandas requires version '2.8.0' or newer of 'numexpr' (version '2.7.3' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [2]:
    with open('config/config_reg.yaml', 'r') as f:
        config = yaml.safe_load(f)
    for k, v in vars(args).items():
        if v is not None and k in config:
            config[k] = v

In [7]:
config["max_depth"] = 3
config["leapfrog_L"] = 1
config["T_max"] = 100
config["step_size"] = 0.01
config["num_samples"] = 100
config["var_height"] = 3

In [8]:
#Generating dataset
from sklearn.model_selection import train_test_split

np.random.seed(42)

n = 500
X = np.random.uniform(-3, 3, size=(n, 1))

# equation
y = 2.5 * X[:, 0] - 1.0

# noise 
y = y + np.random.normal(0, 1.0, size=n)

# 2. train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [9]:
# sampling
samples = sampling(X_train, y_train, config, X_test, y_test, verbose = 1)

Finished feature weight computation!
SAMPLING started ./parameters/normal/abalone/fold0/0308_c*_0.001_d_1_T_100_h-eps_0.01_L_1_nu_10.0_M_1.0
fold 0 1th Trees 	train : 3.15	test : 3.257	 # of Tree :100
fold 0 2th Trees 	train : 1.792	test : 2.017	 # of Tree :100
fold 0 3th Trees 	train : 1.269	test : 1.356	 # of Tree :100
fold 0 4th Trees 	train : 1.071	test : 1.072	 # of Tree :100
fold 0 5th Trees 	train : 1.05	test : 1.081	 # of Tree :100
fold 0 6th Trees 	train : 1.051	test : 1.08	 # of Tree :100
fold 0 7th Trees 	train : 1.034	test : 1.081	 # of Tree :100
fold 0 8th Trees 	train : 1.041	test : 1.157	 # of Tree :100
fold 0 9th Trees 	train : 1.031	test : 1.074	 # of Tree :100
fold 0 10th Trees 	train : 1.012	test : 1.074	 # of Tree :100
fold 0 11th Trees 	train : 1.019	test : 1.12	 # of Tree :100
fold 0 12th Trees 	train : 1.026	test : 1.106	 # of Tree :100
fold 0 13th Trees 	train : 1.024	test : 1.107	 # of Tree :100
fold 0 14th Trees 	train : 1.029	test : 1.097	 # of Tree :100
fold